In [0]:
"""
id: source_0
template: source
templateVersion: 3.0.0
name: media_customer_reviews
position:
  x: 0
  y: 0
previewCodeHash: 315bd0ef510168ac
codegenHash: f836ccce6b5bf6bc
updatedAt: 1792709848462
config:
  table_source:
    tableName: samples.bakehouse.media_customer_reviews
checkpoint:
  mode: auto
input: []
"""

# generated from the system
import atexit
import base64
import csv
import hashlib
import io
import json
import os
import re
import shutil
import tempfile
import time
from typing import Dict, Any, List, Optional
from urllib.parse import parse_qs, unquote, urlparse

_MANAGED_MCP_SERVICES = {
    "google_drive": "system.ai.google_drive",
    "microsoft_365": "system.ai.microsoft_365",
}
_GOOGLE_SHEETS_MIME_TYPE = "application/vnd.google-apps.spreadsheet"
_EXCEL_WORKBOOK_MIME_TYPE = "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
_GOOGLE_DELIMITED_FILE_MIME_TYPES = {
    "application/csv",
    "text/csv",
    "text/tab-separated-values",
}
_MICROSOFT_CSV_FILE_EXTENSION = ".csv"
_MICROSOFT_SAFE_TEXT_PAGE_CHARS = 20_000
_MICROSOFT_CONTENT_TOO_LARGE_ERROR = (
    "Content too large to return even after truncation. "
    "Use smaller limit, offset, or spreadsheet_range for targeted reads."
)
_MANAGED_SOURCE_CACHE_MAX_MEMORY_BYTES = 16 * 1024 * 1024
_MANAGED_SOURCE_CACHE_MAX_DISK_BYTES = 256 * 1024 * 1024
_MANAGED_SOURCE_CACHE_TTL_SECONDS = 15 * 60
_MANAGED_SOURCE_MAX_PAGES = 1_000
_MANAGED_SOURCE_CACHE_GLOBAL = "_lb_mcp_source_payload_cache"
_MANAGED_SOURCE_CACHE_DIRECTORY_GLOBAL = "_lb_mcp_source_payload_cache_directory"

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def _decode_jsonrpc_response(response: Any) -> Dict[str, Any]:
    if isinstance(response, dict) and "contents" in response:
        stream = response["contents"]
        try:
            response = stream.read()
        finally:
            stream.close()
    if isinstance(response, dict):
        return response
    if isinstance(response, bytes):
        response = response.decode("utf-8")
    if not isinstance(response, str):
        raise ValueError("Managed MCP service returned an unsupported response")
    if response.lstrip().startswith("{"):
        parsed = json.loads(response)
        if isinstance(parsed, dict):
            return parsed
    data_parts = [
        line.strip()[len("data:"):].strip()
        for line in response.splitlines()
        if line.strip().startswith("data:")
    ]
    if data_parts:
        parsed = json.loads("\n".join(data_parts))
        if isinstance(parsed, dict) and ("result" in parsed or "error" in parsed):
            return parsed
    raise ValueError("Managed MCP service returned an invalid JSON-RPC response")

def _post_mcp_jsonrpc(api_client, service_name: str, payload: Dict[str, Any]) -> Dict[str, Any]:
    response = api_client.do(
        method="POST",
        path=f"/ai-gateway/mcp-services/{service_name}",
        body=payload,
        headers={
            "Accept": "application/json, text/event-stream",
            "Accept-Encoding": "identity",
            "Content-Type": "application/json",
        },
        raw=True,
    )
    decoded = _decode_jsonrpc_response(response)
    if decoded.get("error"):
        raise ValueError(f"Managed MCP request failed: {decoded['error']}")
    return decoded

def _call_mcp_tool(service_name: str, tool_name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
    from databricks.sdk import WorkspaceClient

    api_client = WorkspaceClient().api_client
    reply = _post_mcp_jsonrpc(
        api_client,
        service_name,
        {
            "jsonrpc": "2.0",
            "id": 1,
            "method": "tools/call",
            "params": {"name": tool_name, "arguments": arguments},
        },
    )
    result = reply.get("result")
    if not isinstance(result, dict):
        raise ValueError("Managed MCP tool returned no result")
    text_parts = [
        item.get("text", "")
        for item in result.get("content", [])
        if isinstance(item, dict) and item.get("type") == "text"
    ]
    if result.get("isError"):
        detail = " ".join(part for part in text_parts if part)
        raise ValueError(f"Managed MCP tool failed: {detail or tool_name}")
    structured = result.get("structuredContent")
    if isinstance(structured, dict):
        return structured
    for text in text_parts:
        try:
            payload = json.loads(text)
        except (TypeError, json.JSONDecodeError):
            continue
        if isinstance(payload, dict):
            return payload
    raise ValueError("Managed MCP tool returned no structured spreadsheet content")

def _managed_source_cache_identity():
    try:
        from databricks.sdk import WorkspaceClient

        client = WorkspaceClient()
        current_user = client.current_user.me()
        user_id = getattr(current_user, "id", None)
        host = getattr(client.config, "host", None)
        if user_id and host:
            return (str(host), str(user_id))
    except Exception:
        pass
    return None

def _managed_source_cache_key(identity, *parts: Any) -> str:
    return json.dumps((identity, *parts), ensure_ascii=False, separators=(",", ":"), default=str)

def _managed_source_cache_remove(cache: Dict[str, Any], key: str) -> None:
    entry = cache.pop(key, None)
    path = entry.get("path") if isinstance(entry, dict) else None
    if isinstance(path, str):
        try:
            os.remove(path)
        except OSError:
            pass

def _managed_source_cache_directory() -> str:
    existing = globals().get(_MANAGED_SOURCE_CACHE_DIRECTORY_GLOBAL)
    if isinstance(existing, str) and os.path.isdir(existing):
        return existing
    directory = tempfile.mkdtemp(prefix="lakebuilder-mcp-")
    globals()[_MANAGED_SOURCE_CACHE_DIRECTORY_GLOBAL] = directory
    atexit.register(shutil.rmtree, directory, ignore_errors=True)
    return directory

def _managed_source_cache_cleanup(cache: Dict[str, Any], now: float) -> None:
    expired_keys = [
        key
        for key, entry in cache.items()
        if not isinstance(entry, dict)
        or now - entry.get("last_accessed", 0) >= _MANAGED_SOURCE_CACHE_TTL_SECONDS
    ]
    for key in expired_keys:
        _managed_source_cache_remove(cache, key)

def _managed_source_cache_get(key: str, version: Any = None):
    if not globals().get("_lb_mcp_preview_cache_enabled", False):
        return None
    cache = globals().setdefault(_MANAGED_SOURCE_CACHE_GLOBAL, {})
    now = time.monotonic()
    _managed_source_cache_cleanup(cache, now)
    entry = cache.get(key)
    if not isinstance(entry, dict):
        cache.pop(key, None)
        return None
    if globals().get("_lb_force", False) or entry.get("version") != version:
        _managed_source_cache_remove(cache, key)
        return None
    value = entry.get("value")
    if isinstance(value, dict):
        entry["last_accessed"] = now
        return value
    path = entry.get("path")
    if isinstance(path, str):
        try:
            with open(path, encoding="utf-8") as handle:
                value = json.load(handle)
                if isinstance(value, dict):
                    entry["last_accessed"] = now
                    return value
        except (OSError, json.JSONDecodeError):
            pass
    _managed_source_cache_remove(cache, key)
    return None

def _managed_source_cache_put(key: str, value: Dict[str, Any], version: Any = None) -> None:
    if not globals().get("_lb_mcp_preview_cache_enabled", False):
        return
    encoded = json.dumps(value, ensure_ascii=False, default=str).encode("utf-8")
    cache = globals().setdefault(_MANAGED_SOURCE_CACHE_GLOBAL, {})
    now = time.monotonic()
    _managed_source_cache_cleanup(cache, now)
    _managed_source_cache_remove(cache, key)
    memory_size = sum(
        entry.get("size", 0)
        for entry in cache.values()
        if isinstance(entry, dict) and isinstance(entry.get("value"), dict)
    )
    if memory_size + len(encoded) <= _MANAGED_SOURCE_CACHE_MAX_MEMORY_BYTES:
        cache[key] = {
            "size": len(encoded),
            "value": value,
            "version": version,
            "last_accessed": now,
        }
        return
    if len(encoded) > _MANAGED_SOURCE_CACHE_MAX_DISK_BYTES:
        return
    disk_entries = sorted(
        (
            (entry.get("last_accessed", 0), entry_key, entry.get("size", 0))
            for entry_key, entry in cache.items()
            if isinstance(entry, dict) and isinstance(entry.get("path"), str)
        ),
    )
    disk_size = sum(size for _, _, size in disk_entries)
    for _, entry_key, size in disk_entries:
        if disk_size + len(encoded) <= _MANAGED_SOURCE_CACHE_MAX_DISK_BYTES:
            break
        _managed_source_cache_remove(cache, entry_key)
        disk_size -= size
    temporary_path = None
    try:
        directory = _managed_source_cache_directory()
        path = os.path.join(directory, hashlib.sha256(key.encode("utf-8")).hexdigest() + ".json")
        file_descriptor, temporary_path = tempfile.mkstemp(prefix=".tmp-", dir=directory)
        with os.fdopen(file_descriptor, "wb") as handle:
            handle.write(encoded)
        os.replace(temporary_path, path)
    except OSError:
        if temporary_path is not None:
            try:
                os.remove(temporary_path)
            except OSError:
                pass
        return
    cache[key] = {
        "size": len(encoded),
        "path": path,
        "version": version,
        "last_accessed": now,
    }

def _extract_google_file_id(file_url: str) -> str:
    file_url = file_url.strip()
    match = re.search(r"/(?:spreadsheets|file)(?:/u/[0-9]+)?/d/([^/?#]+)", file_url)
    if match:
        return match.group(1)
    query_id = parse_qs(urlparse(file_url).query).get("id", [None])[0]
    if query_id:
        return query_id
    if re.fullmatch(r"[A-Za-z0-9_-]{5,}", file_url):
        return file_url
    raise ValueError("Google Drive source requires a valid Google Drive URL or file ID")

def _collect_sharepoint_candidates(value: Any, candidates: List[Dict[str, str]]) -> None:
    if isinstance(value, dict):
        parent = value.get("parentReference")
        drive_id = parent.get("driveId") if isinstance(parent, dict) else None
        item_id = value.get("id")
        if drive_id and item_id:
            candidates.append({
                "drive_id": str(drive_id),
                "item_id": str(item_id),
                "name": str(value.get("name") or ""),
                "web_url": str(value.get("webUrl") or ""),
            })
        for child in value.values():
            _collect_sharepoint_candidates(child, candidates)
    elif isinstance(value, list):
        for child in value:
            _collect_sharepoint_candidates(child, candidates)

def _collect_sharepoint_search_candidates(value: Any, candidates: List[Dict[str, str]]) -> None:
    if not isinstance(value, dict):
        return
    for search_response in value.get("value", []):
        if not isinstance(search_response, dict):
            continue
        for container in search_response.get("hitsContainers", []):
            if not isinstance(container, dict):
                continue
            for hit in container.get("hits", []):
                if not isinstance(hit, dict):
                    continue
                resource = hit.get("resource")
                if isinstance(resource, dict):
                    parent = resource.get("parentReference")
                    drive_id = parent.get("driveId") if isinstance(parent, dict) else None
                    item_id = resource.get("id")
                    if drive_id and item_id:
                        candidates.append({
                            "drive_id": str(drive_id),
                            "item_id": str(item_id),
                            "name": str(resource.get("name") or ""),
                            "web_url": str(resource.get("webUrl") or ""),
                        })

def _normalized_url_path(value: str) -> str:
    parsed = urlparse(value)
    return unquote(parsed.path).rstrip("/").casefold()

def _sharepoint_search_name(file_url: str) -> str:
    parsed = urlparse(file_url)
    query = parse_qs(parsed.query)
    for key in ("id", "file", "filename"):
        value = query.get(key, [None])[0]
        if value:
            name = unquote(value).rstrip("/").split("/")[-1]
            if name:
                return name
    name = unquote(parsed.path).rstrip("/").split("/")[-1]
    if name.casefold() not in {"allitems.aspx", "doc.aspx", "view.aspx"} and name:
        return name
    raise ValueError("SharePoint source URL does not identify a supported file; use its Preview URL")

def _microsoft_share_id(file_url: str) -> str:
    parsed = urlparse(file_url)
    host = (parsed.hostname or "").casefold()
    sharepoint_url = host.endswith(".sharepoint.com") and bool(
        re.match(r"^/:[a-z]:/", parsed.path, re.IGNORECASE)
    )
    onedrive_url = host == "1drv.ms" or host.endswith(".1drv.ms") or host == "onedrive.live.com"
    if not sharepoint_url and not onedrive_url:
        return ""
    encoded = base64.urlsafe_b64encode(file_url.encode("utf-8")).decode("ascii").rstrip("=")
    return f"u!{encoded}"

def _resource_uri_from_drive_item(value: Any, error_message: str) -> str:
    candidates: List[Dict[str, str]] = []
    _collect_sharepoint_candidates(value, candidates)
    deduplicated = {
        (candidate["drive_id"], candidate["item_id"]): candidate
        for candidate in candidates
    }
    if len(deduplicated) != 1:
        raise ValueError(error_message)
    drive_id, item_id = next(iter(deduplicated))
    return f"/v1.0/drives/{drive_id}/items/{item_id}/content"

def _resolve_microsoft_resource_uri(file_url: str) -> str:
    direct = re.search(r"/v1\.0/drives/([^/]+)/items/([^/?#]+)", file_url)
    if direct:
        return f"/v1.0/drives/{direct.group(1)}/items/{direct.group(2)}/content"

    parsed = urlparse(file_url)
    query = {key.casefold(): values for key, values in parse_qs(parsed.query).items()}
    drive_id = query.get("driveid", [None])[0]
    item_id = query.get("itemid", [None])[0]
    if drive_id and item_id:
        return f"/v1.0/drives/{drive_id}/items/{item_id}/content"

    share_id = _microsoft_share_id(file_url)
    if share_id:
        drive_item = _call_mcp_tool(
            "system.ai.microsoft_365",
            "read_resource",
            {"resource_uri": f"/v1.0/shares/{share_id}/driveItem"},
        )
        return _resource_uri_from_drive_item(
            drive_item,
            "Could not resolve the SharePoint or OneDrive sharing URL to a supported file.",
        )

    file_name = _sharepoint_search_name(file_url)
    search_payload = _call_mcp_tool(
        "system.ai.microsoft_365",
        "sharepoint_search",
        {"query": file_name, "max_results": 100},
    )
    candidates: List[Dict[str, str]] = []
    _collect_sharepoint_search_candidates(search_payload, candidates)
    deduplicated = {
        (candidate["drive_id"], candidate["item_id"]): candidate
        for candidate in candidates
    }
    candidates = list(deduplicated.values())
    input_path = _normalized_url_path(file_url)
    exact = [
        candidate
        for candidate in candidates
        if candidate["web_url"]
        and (
            input_path.endswith(_normalized_url_path(candidate["web_url"]))
            or _normalized_url_path(candidate["web_url"]).endswith(input_path)
        )
    ]
    matches = exact or [candidate for candidate in candidates if candidate["name"].casefold() == file_name.casefold()]
    if len(matches) != 1:
        raise ValueError(
            "Could not uniquely resolve the SharePoint file URL. Open the file in Preview and copy the browser URL."
        )
    match = matches[0]
    return f"/v1.0/drives/{match['drive_id']}/items/{match['item_id']}/content"

def _read_google_delimited_source(service_name: str, file_id: str) -> Dict[str, Any]:
    chunks: List[str] = []
    offset = 0
    metadata: Dict[str, Any] = {}
    for _ in range(_MANAGED_SOURCE_MAX_PAGES):
        try:
            payload = _call_mcp_tool(
                service_name,
                "google_file_read",
                {"file_id": file_id, "file_type": "csv", "limit": -1, "offset": offset},
            )
        except ValueError:
            if offset == 0:
                return _call_mcp_tool(service_name, "google_file_download", {"file_id": file_id})
            raise
        page_start = payload.get("start_char_offset")
        delivered = payload.get("reading_length_in_chars")
        if offset == 0 and page_start is None and delivered is None:
            return _call_mcp_tool(service_name, "google_file_download", {"file_id": file_id})
        content = payload.get("content")
        if not isinstance(content, str):
            raise ValueError("Managed MCP delimited-text response has no text content")
        truncated = payload.get("truncated")
        if page_start != offset or delivered != len(content) or not isinstance(truncated, bool):
            raise ValueError("Managed MCP delimited-text response has invalid pagination metadata")
        page_metadata = payload.get("metadata")
        if isinstance(page_metadata, dict):
            metadata = page_metadata
        chunks.append(content)
        if not truncated:
            return {
                "content": "".join(chunks),
                "name": metadata.get("name"),
                "mime_type": metadata.get("mime_type"),
                "truncated": False,
            }
        if delivered == 0:
            raise ValueError("Managed MCP delimited-text pagination made no progress")
        offset += delivered
    raise ValueError("Managed MCP delimited-text read exceeded the pagination limit")

def _sheet_name(range_name: str) -> str:
    name = range_name.split("!", 1)[0].strip()
    if len(name) >= 2 and name[0] == "'" and name[-1] == "'":
        name = name[1:-1].replace("''", "'")
    return name

def _target_spreadsheet_sheet(payload: Dict[str, Any], file_url: str):
    gid = parse_qs(urlparse(file_url).fragment).get("gid", [None])[0]
    metadata = payload.get("metadata")
    tabs = metadata.get("sheet_tabs", []) if isinstance(metadata, dict) else []
    if gid:
        for tab in tabs:
            if isinstance(tab, dict) and str(tab.get("sheetId")) == str(gid):
                return tab.get("title")
    return None

def _read_microsoft_spreadsheet_source(
    service_name: str,
    arguments: Dict[str, Any],
    initial_payload: Dict[str, Any],
    file_url: str,
) -> Dict[str, Any]:
    pages: List[List[Any]] = []
    range_name = None
    offset = 0
    first_page = None
    metadata_version = None
    read_arguments = dict(arguments)
    pending_payload = initial_payload
    target_sheet = _target_spreadsheet_sheet(initial_payload, file_url)
    if target_sheet is not None and "spreadsheet_range" not in read_arguments:
        read_arguments["spreadsheet_range"] = target_sheet
        pending_payload = None
    for _ in range(_MANAGED_SOURCE_MAX_PAGES):
        if pending_payload is not None:
            payload = pending_payload
            pending_payload = None
        else:
            payload = _call_mcp_tool(
                service_name,
                "read_resource",
                {**read_arguments, "content_offset": offset},
            )
        if payload.get("error") or payload.get("error_code"):
            raise ValueError(f"Managed MCP spreadsheet read failed: {payload}")
        truncated = payload.get("truncated")
        metadata = payload.get("metadata")
        if isinstance(metadata, dict):
            page_metadata_version = (
                metadata.get("id"),
                metadata.get("size"),
                metadata.get("lastModifiedDateTime"),
            )
            if metadata_version is None:
                metadata_version = page_metadata_version
            elif page_metadata_version != metadata_version:
                raise ValueError("Managed Microsoft 365 workbook changed during pagination")
        elif metadata_version is not None:
            raise ValueError("Managed MCP spreadsheet response has invalid pagination metadata")
        if truncated is False and first_page is None:
            return payload
        content = payload.get("content")
        if not isinstance(content, dict) or not isinstance(metadata, dict):
            raise ValueError(
                "Managed MCP spreadsheet content was truncated without valid pagination metadata. "
                "Configure spreadsheetRange to read a single range."
            )
        sheet_names = metadata.get("sheet_names")
        if (
            first_page is None
            and "spreadsheet_range" not in read_arguments
            and (len(content) != 1 or isinstance(sheet_names, list) and len(sheet_names) > 1)
        ):
            selected_range = next(
                (
                    page_range
                    for page_range, page_content in content.items()
                    if isinstance(page_content, dict)
                    and not page_content.get("error")
                    and isinstance(page_content.get("values"), list)
                    and page_content["values"]
                ),
                None,
            )
            if selected_range is None:
                raise ValueError("Managed MCP spreadsheet contains no readable sheet rows")
            read_arguments["spreadsheet_range"] = _sheet_name(str(selected_range))
            continue
        if len(content) != 1:
            raise ValueError(
                "Managed MCP spreadsheet content was truncated without valid pagination metadata. "
                "Configure spreadsheetRange to read a single range."
            )
        page_range, page_content = next(iter(content.items()))
        values = page_content.get("values") if isinstance(page_content, dict) else None
        next_offset = metadata.get("next_offset")
        has_more = metadata.get("has_more")
        if (
            not isinstance(values, list)
            or payload.get("offset") != offset
            or isinstance(next_offset, bool)
            or not isinstance(next_offset, int)
            or next_offset != offset + len(values)
            or not isinstance(has_more, bool)
            or has_more != truncated
            or (range_name is not None and page_range != range_name)
        ):
            raise ValueError("Managed MCP spreadsheet response has invalid pagination metadata")
        if first_page is None:
            first_page = payload
            range_name = page_range
        pages.extend(values)
        if not has_more:
            combined = {**first_page, **payload}
            first_content = first_page["content"][range_name]
            combined_content = {
                range_name: {
                    **first_content,
                    **page_content,
                    "values": pages,
                    "row_count": len(pages),
                }
            }
            combined["content"] = combined_content
            combined_metadata = {
                **first_page["metadata"],
                **metadata,
                "content_length": len(json.dumps(combined_content, indent=2)),
                "returned_rows": len(pages),
            }
            combined["metadata"] = combined_metadata
            combined["truncated"] = False
            combined["offset"] = 0
            combined.pop("hint", None)
            return combined
        if next_offset <= offset:
            raise ValueError("Managed MCP spreadsheet pagination made no progress")
        offset = next_offset
    raise ValueError("Managed MCP spreadsheet read exceeded the pagination limit")

def _read_microsoft_source(
    service_name: str,
    resource_uri: str,
    spreadsheet_range: Optional[str],
    file_url: str,
):
    chunks: List[str] = []
    offset = 0
    content_limit = -1
    metadata: Dict[str, Any] = {}
    metadata_version = None
    for _ in range(_MANAGED_SOURCE_MAX_PAGES):
        arguments: Dict[str, Any] = {
            "resource_uri": resource_uri,
            "content_limit": content_limit,
            "content_offset": offset,
        }
        if spreadsheet_range:
            arguments["spreadsheet_range"] = spreadsheet_range
        payload = _call_mcp_tool(service_name, "read_resource", arguments)
        if payload.get("error") or payload.get("error_code"):
            raise ValueError(f"Managed MCP file read failed: {payload}")
        content = payload.get("content")
        if isinstance(content, dict):
            if offset != 0:
                raise ValueError("Managed Microsoft 365 response changed content type during pagination")
            return "spreadsheet", _read_microsoft_spreadsheet_source(
                service_name,
                arguments,
                payload,
                file_url,
            )
        if not isinstance(content, str):
            raise ValueError("Managed Microsoft 365 file response has no supported content")

        page_metadata = payload.get("metadata")
        if isinstance(page_metadata, dict):
            page_metadata_version = (
                page_metadata.get("id"),
                page_metadata.get("size"),
                page_metadata.get("lastModifiedDateTime"),
            )
            if metadata_version is None:
                metadata_version = page_metadata_version
            elif page_metadata_version != metadata_version:
                raise ValueError("Managed Microsoft 365 file changed during pagination")
            metadata = page_metadata
        file_name = str(metadata.get("name") or "")
        if not file_name.casefold().endswith(_MICROSOFT_CSV_FILE_EXTENSION):
            raise ValueError("Managed Microsoft 365 sources support Excel and CSV files")
        if spreadsheet_range:
            raise ValueError("Managed Microsoft 365 CSV sources do not support spreadsheetRange")

        decode_error = (
            f"File '{file_name}' could not be decoded as UTF-8 text. "
            "The file may use a different encoding. "
            "Please access it directly via the web link."
        )
        if content == decode_error:
            raise ValueError("Managed Microsoft 365 CSV source must be UTF-8 encoded")
        if content == _MICROSOFT_CONTENT_TOO_LARGE_ERROR:
            if content_limit == -1:
                content_limit = _MICROSOFT_SAFE_TEXT_PAGE_CHARS
                continue
            raise ValueError("Managed Microsoft 365 CSV page exceeds the response size limit")

        page_start = payload.get("offset")
        truncated = payload.get("truncated")
        if page_start != offset or not isinstance(truncated, bool):
            raise ValueError("Managed MCP delimited-text response has invalid pagination metadata")
        chunks.append(content)
        if not truncated:
            return "delimited_text", {
                "metadata": metadata,
                "content": "".join(chunks),
                "name": file_name,
                "truncated": False,
            }
        if not content:
            raise ValueError("Managed MCP delimited-text pagination made no progress")
        offset += len(content)
    raise ValueError("Managed MCP delimited-text read exceeded the pagination limit")

def _read_google_excel_source(
    service_name: str, arguments: Dict[str, Any]
) -> Dict[str, Any]:
    read_arguments = dict(arguments)
    payload = _call_mcp_tool(service_name, "google_file_read", read_arguments)
    metadata = payload.get("metadata")
    if not isinstance(metadata, dict):
        return payload

    has_more_present = "has_more" in metadata
    next_offset_present = "next_offset" in metadata
    if not has_more_present and not next_offset_present:
        content = payload.get("content")
        if (
            payload.get("truncated") is not True
            or "spreadsheet_range" in read_arguments
            or not isinstance(content, dict)
        ):
            return payload
        selected_range = next(
            (
                page_range
                for page_range, page_content in content.items()
                if isinstance(page_content, dict)
                and not page_content.get("error")
                and isinstance(page_content.get("values"), list)
                and page_content["values"]
            ),
            None,
        )
        if selected_range is None:
            raise ValueError("Managed MCP Excel workbook contains no readable sheet rows")
        read_arguments["spreadsheet_range"] = selected_range
        read_arguments["offset"] = 0
        payload = _call_mcp_tool(service_name, "google_file_read", read_arguments)
        metadata = payload.get("metadata")
        if not isinstance(metadata, dict):
            return payload

    has_more = metadata.get("has_more")
    next_offset = metadata.get("next_offset")
    content = payload.get("content")
    if (
        not isinstance(has_more, bool)
        or not isinstance(next_offset, int)
        or isinstance(next_offset, bool)
        or not isinstance(content, dict)
        or len(content) != 1
    ):
        raise ValueError("Managed MCP Excel response has invalid pagination metadata")

    range_name, aggregate_entry = next(iter(content.items()))
    if not isinstance(aggregate_entry, dict) or not isinstance(aggregate_entry.get("values"), list):
        raise ValueError("Managed MCP Excel response has invalid pagination content")
    aggregate_values = aggregate_entry["values"]
    if next_offset != len(aggregate_values):
        raise ValueError("Managed MCP Excel pagination has a gap or overlap")
    if not has_more:
        return payload

    offset = next_offset
    for _ in range(_MANAGED_SOURCE_MAX_PAGES - 1):
        page_arguments = dict(read_arguments)
        page_arguments["offset"] = offset
        page = _call_mcp_tool(service_name, "google_file_read", page_arguments)
        if page.get("error") or page.get("error_code"):
            raise ValueError(f"Managed MCP spreadsheet read failed: {page}")

        page_metadata = page.get("metadata")
        page_content = page.get("content")
        if not isinstance(page_metadata, dict) or not isinstance(page_content, dict):
            raise ValueError("Managed MCP Excel response has invalid pagination metadata")
        page_has_more = page_metadata.get("has_more")
        page_next_offset = page_metadata.get("next_offset")
        page_entry = page_content.get(range_name)
        page_values = page_entry.get("values") if isinstance(page_entry, dict) else None
        if (
            not isinstance(page_has_more, bool)
            or not isinstance(page_next_offset, int)
            or isinstance(page_next_offset, bool)
            or not isinstance(page_values, list)
            or page_next_offset != offset + len(page_values)
        ):
            raise ValueError("Managed MCP Excel pagination has a gap or overlap")

        aggregate_values.extend(page_values)
        aggregate_entry["row_count"] = len(aggregate_values)
        metadata.update(page_metadata)
        if not page_has_more:
            metadata["reading_length_in_chars"] = len(json.dumps(content, indent=2))
            payload["truncated"] = False
            return payload
        if page_next_offset == offset:
            raise ValueError("Managed MCP Excel pagination made no progress")
        offset = page_next_offset

    raise ValueError("Managed MCP Excel read exceeded the pagination limit")

def _read_managed_source(config: Dict[str, Any]) -> Dict[str, Any]:
    provider = config.get("provider")
    service_name = config.get("serviceName")
    file_url = config.get("fileUrl")
    spreadsheet_range = config.get("spreadsheetRange")
    expected_service = _MANAGED_MCP_SERVICES.get(provider)
    if not expected_service or service_name != expected_service:
        raise ValueError("Source: unsupported managed MCP provider or service")
    if not file_url:
        raise ValueError("Source: 'fileUrl' is required for managed MCP source")
    cache_identity = (
        _managed_source_cache_identity()
        if globals().get("_lb_mcp_preview_cache_enabled", False)
        else None
    )

    if provider == "google_drive":
        file_id = _extract_google_file_id(file_url)
        try:
            metadata = _call_mcp_tool(service_name, "google_file_metadata", {"file_id": file_id})
        except ValueError as error:
            detail = str(error).casefold()
            if "404" in detail or "not found" in detail:
                raise ValueError(
                    "Google Drive file was not found or is not accessible to the connected Google account. "
                    "Reconnect with an account that can access the file, or update its sharing permissions."
                ) from error
            raise
        mime_type = str(metadata.get("mimeType") or "").casefold()
        file_name = str(metadata.get("name") or "").casefold()
        cache_version = (
            metadata.get("version"),
            metadata.get("modifiedTime"),
            metadata.get("md5Checksum"),
        )
        if mime_type in {_GOOGLE_SHEETS_MIME_TYPE, _EXCEL_WORKBOOK_MIME_TYPE}:
            arguments: Dict[str, Any] = {
                "file_id": file_id,
                "file_type": "spreadsheet",
                "limit": -1,
            }
            if spreadsheet_range:
                arguments["spreadsheet_range"] = spreadsheet_range
            source_kind = "spreadsheet"
            tool_name = "google_file_read"
        elif mime_type in _GOOGLE_DELIMITED_FILE_MIME_TYPES or file_name.endswith((".csv", ".tsv")):
            if spreadsheet_range:
                raise ValueError("Managed Google Drive CSV and TSV sources do not support spreadsheetRange")
            source_kind = "delimited_text"
            tool_name = "google_file_read"
            arguments = {"file_id": file_id, "file_type": "csv", "limit": -1, "offset": 0}
        else:
            raise ValueError("Managed Google Drive sources support Google Sheets, Excel, CSV, and TSV files")
        cache_key = _managed_source_cache_key(
            cache_identity,
            provider,
            service_name,
            file_id,
            spreadsheet_range,
            source_kind,
        )
    else:
        resource_uri = _resolve_microsoft_resource_uri(file_url)
        source_kind = "microsoft_file"
        cache_key = _managed_source_cache_key(
            cache_identity,
            provider,
            service_name,
            resource_uri,
            spreadsheet_range,
            source_kind,
        )
        cache_version = None

    cached = _managed_source_cache_get(cache_key, cache_version) if cache_identity is not None else None
    if cached is not None:
        return cached
    if provider == "google_drive" and source_kind == "delimited_text":
        payload = _read_google_delimited_source(service_name, file_id)
    elif provider == "google_drive" and mime_type == _EXCEL_WORKBOOK_MIME_TYPE:
        payload = _read_google_excel_source(service_name, arguments)
    elif provider == "microsoft_365":
        source_kind, payload = _read_microsoft_source(
            service_name,
            resource_uri,
            spreadsheet_range,
            file_url,
        )
    else:
        payload = _call_mcp_tool(service_name, tool_name, arguments)

    if payload.get("error") or payload.get("error_code"):
        raise ValueError(f"Managed MCP spreadsheet read failed: {payload}")
    if payload.get("truncated"):
        raise ValueError(
            "Managed MCP spreadsheet content was truncated. Configure spreadsheetRange to read a smaller range."
        )
    result = {"kind": source_kind, "payload": payload}
    if cache_identity is not None:
        _managed_source_cache_put(cache_key, result, cache_version)
    return result

def _select_spreadsheet_values(payload: Dict[str, Any], file_url: str) -> List[List[Any]]:
    content = payload.get("content")
    if not isinstance(content, dict):
        raise ValueError("Managed MCP spreadsheet response has no sheet content")

    target_sheet = _target_spreadsheet_sheet(payload, file_url)

    candidates = []
    for range_name, sheet in content.items():
        if not isinstance(sheet, dict):
            continue
        if sheet.get("error"):
            continue
        values = sheet.get("values")
        if isinstance(values, list) and values:
            candidates.append((str(range_name), values))
    if target_sheet:
        for range_name, values in candidates:
            if _sheet_name(range_name) == target_sheet:
                return values
        raise ValueError(f"Managed MCP response did not contain the selected sheet: {target_sheet}")
    if not candidates:
        raise ValueError("Managed MCP spreadsheet contains no rows")
    return candidates[0][1]

def _normalized_headers(header_row: List[Any], width: int) -> List[str]:
    headers: List[str] = []
    used = set()
    for index in range(width):
        raw = header_row[index] if index < len(header_row) else None
        base = str(raw).strip() if raw is not None else ""
        if not base:
            base = f"_c{index + 1}"
        candidate = base
        suffix = 2
        while candidate.casefold() in used:
            candidate = f"{base}_{suffix}"
            suffix += 1
        headers.append(candidate)
        used.add(candidate.casefold())
    return headers

def _values_dataframe(values: List[List[Any]], spark):
    from pyspark.sql.types import BooleanType, DoubleType, LongType, StringType, StructField, StructType

    if not all(isinstance(row, list) for row in values):
        raise ValueError("Managed MCP tabular source returned malformed rows")
    width = max((len(row) for row in values), default=0)
    if width == 0:
        raise ValueError("Managed MCP tabular source contains no columns")
    headers = _normalized_headers(values[0], width)
    columns = [
        [row[index] if index < len(row) else None for row in values[1:]]
        for index in range(width)
    ]
    fields = []
    normalized_columns = []
    for header, column in zip(headers, columns):
        present = [value for value in column if value is not None]
        if present and all(isinstance(value, bool) for value in present):
            data_type = BooleanType()
            normalized = column
        elif present and all(isinstance(value, int) and not isinstance(value, bool) for value in present):
            data_type = LongType()
            normalized = column
        elif present and all(
            isinstance(value, (int, float)) and not isinstance(value, bool)
            for value in present
        ):
            data_type = DoubleType()
            normalized = [None if value is None else float(value) for value in column]
        elif not present or all(isinstance(value, str) for value in present):
            data_type = StringType()
            normalized = column
        else:
            data_type = StringType()
            normalized = [None if value is None else str(value) for value in column]
        fields.append(StructField(header, data_type, True))
        normalized_columns.append(normalized)
    rows = [list(row) for row in zip(*normalized_columns)]
    schema = StructType(fields)
    return spark.createDataFrame(rows, schema=schema)

def _managed_source_dataframe(result: Dict[str, Any], file_url: str, spark):
    payload = result.get("payload")
    if not isinstance(payload, dict):
        raise ValueError("Managed MCP source returned no content")
    if result.get("kind") == "spreadsheet":
        values = _select_spreadsheet_values(payload, file_url)
    elif result.get("kind") == "delimited_text":
        content = payload.get("content")
        if not isinstance(content, str) or not content:
            raise ValueError("Managed MCP file contains no text")
        delimiter = "	" if str(payload.get("name") or "").casefold().endswith(".tsv") else ","
        try:
            dialect = csv.Sniffer().sniff(content[:4096], delimiters=",	;|")
            values = list(csv.reader(io.StringIO(content.lstrip("﻿")), dialect))
        except csv.Error:
            values = list(csv.reader(io.StringIO(content.lstrip("﻿")), delimiter=delimiter))
    else:
        raise ValueError("Managed MCP source returned an unsupported content type")
    return _values_dataframe(values, spark)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")
    mcp_file_source = config.get("mcp_file_source")
    source_modes = [file_source, table_source, mcp_file_source]
    if sum(mode is not None for mode in source_modes) != 1:
        raise ValueError("Source: exactly one of 'file_source', 'table_source', or 'mcp_file_source' must be configured")

    if file_source is not None:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source is not None:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    elif mcp_file_source is not None:
        result = _read_managed_source(mcp_file_source)
        out = _managed_source_dataframe(result, mcp_file_source.get("fileUrl", ""), spark)

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "samples.bakehouse.media_customer_reviews"
    }
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["source_0.data"] = out["data"]
if globals().get("ld_display_outputs", False) or "source_0" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["source_0.data"])
if "source_0" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "source_0", "counts": {"data": int(ctx["source_0.data"].count())}}}, raw=True)